# 05 — SCIN Dataset: Gaussian Noise Augmentation

Downloads SCIN images, creates Gaussian-noised copies, and uploads
original + noised images to our GCS bucket.

**What this does:**
1. Download SCIN images from public bucket → **verify** valid count
2. Add Gaussian noise (std=25, seed=42) → **verify** noise stats
3. Upload to our bucket → **verify** upload count matches

All noise logic is imported from `scripts/add_gaussian_noise.py` — no duplication.

**References:**
- SCIN dataset: https://github.com/google-research-datasets/scin
- Augmentation: https://pmc.ncbi.nlm.nih.gov/articles/PMC5977656/

In [ ]:
# Cell 1: Setup
import os, subprocess, sys

if not os.path.exists("/content/NST_Class"):
    subprocess.run(["git", "clone", "https://github.com/AayushBaniya2006/NST_Class.git"], cwd="/content")
else:
    subprocess.run(["git", "pull"], cwd="/content/NST_Class")
os.chdir("/content/NST_Class")

!pip install -q -r requirements.txt
sys.path.insert(0, "/content/NST_Class")

print("Setup complete!")

In [ ]:
# Cell 2: Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Authenticated!")

In [ ]:
# Cell 3: Configuration
SOURCE_BUCKET = "dx-scin-public-data"
SOURCE_PREFIX = "dataset/images"
DEST_BUCKET = "skin-tone-project"
DEST_PREFIX = "scin_images"

LOCAL_IMAGE_DIR = "data/scin/images"
LOCAL_NOISED_DIR = "data/scin/images_noised"

NOISE_MEAN = 0.0
NOISE_STD = 25.0
SEED = 42

print(f"Source: gs://{SOURCE_BUCKET}/{SOURCE_PREFIX}")
print(f"Dest:   gs://{DEST_BUCKET}/{DEST_PREFIX}")
print(f"Noise:  mean={NOISE_MEAN}, std={NOISE_STD}, seed={SEED}")

In [ ]:
# Cell 4: Download SCIN images (gcloud parallel) + verify
from scripts.add_gaussian_noise import verify_download

!mkdir -p {LOCAL_IMAGE_DIR}
!gcloud storage cp -r -n gs://{SOURCE_BUCKET}/{SOURCE_PREFIX}/ {LOCAL_IMAGE_DIR}/
# gcloud storage is Go-based — much faster than gsutil for bulk transfers
# -n = skip existing

dl_result = verify_download(LOCAL_IMAGE_DIR)
print(f"\nVerification: {dl_result['valid']} valid, {dl_result['corrupt']} corrupt out of {dl_result['total']}")
assert dl_result["valid"] > 0, "No valid images found!"

In [ ]:
# Cell 5: Add Gaussian noise
from scripts.add_gaussian_noise import process_images

count_before = dl_result["valid"]
created, noised_paths = process_images(
    LOCAL_IMAGE_DIR, LOCAL_NOISED_DIR,
    mean=NOISE_MEAN, std=NOISE_STD, seed=SEED,
)

print(f"\nOriginals:  {count_before}")
print(f"Created:    {created} noised images")
assert created > 0, "No noised images created!"

In [ ]:
# Cell 6: Verify noise quality + visual spot-check
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path
from scripts.add_gaussian_noise import verify_noise

noise_result = verify_noise(LOCAL_IMAGE_DIR, LOCAL_NOISED_DIR, expected_std=NOISE_STD)
print(f"Passed:          {noise_result['passed']}")
print(f"Pairs checked:   {noise_result['checked']}")
print(f"Missing:         {len(noise_result['missing'])}")
print(f"Mean noise mean: {noise_result['mean_noise_mean']:.2f} (expect ~0)")
print(f"Mean noise std:  {noise_result['mean_noise_std']:.2f} (expect ~{NOISE_STD})")
assert noise_result["passed"], f"Noise verification failed: {noise_result}"

# Visual: show 3 original vs noised pairs side-by-side
from src.data.prepare import _IMAGE_EXTENSIONS
_ext_set = {e.lower() for e in _IMAGE_EXTENSIONS}
originals = sorted(
    f for f in Path(LOCAL_IMAGE_DIR).rglob("*")
    if f.is_file() and f.suffix.lower() in _ext_set and "_noised" not in f.stem
)[:3]

fig, axes = plt.subplots(len(originals), 2, figsize=(8, 4 * len(originals)))
if len(originals) == 1:
    axes = [axes]
for i, orig in enumerate(originals):
    noised_path = Path(LOCAL_NOISED_DIR) / f"{orig.stem}_noised{orig.suffix}"
    axes[i][0].imshow(Image.open(orig))
    axes[i][0].set_title(f"Original: {orig.name}")
    axes[i][0].axis("off")
    if noised_path.exists():
        axes[i][1].imshow(Image.open(noised_path))
        axes[i][1].set_title(f"Noised: {noised_path.name}")
    axes[i][1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Upload to bucket (gcloud parallel) + verify
from scripts.add_gaussian_noise import verify_upload

# Upload originals + noised
!gcloud storage cp -r {LOCAL_IMAGE_DIR}/ gs://{DEST_BUCKET}/{DEST_PREFIX}/originals/
!gcloud storage cp -r {LOCAL_NOISED_DIR}/ gs://{DEST_BUCKET}/{DEST_PREFIX}/noised/

# Verify
expected_count = dl_result["valid"] + created
upload_result = verify_upload(DEST_BUCKET, DEST_PREFIX, expected_count)
print(f"\nUpload verification: {upload_result['actual_count']}/{upload_result['expected_count']} files")
assert upload_result["passed"], f"Upload verification failed: {upload_result}"

In [ ]:
# Cell 8: Summary
print("=" * 60)
print("AUGMENTATION PIPELINE SUMMARY")
print("=" * 60)
print(f"Download:  {dl_result['valid']} valid images ({dl_result['corrupt']} corrupt)")
print(f"Noise:     {created} noised images (std={NOISE_STD}, seed={SEED})")
print(f"Upload:    {upload_result['actual_count']} files in gs://{DEST_BUCKET}/{DEST_PREFIX}")
print(f"")
print(f"Verify download: PASS ({dl_result['valid']} valid)")
print(f"Verify noise:    {'PASS' if noise_result['passed'] else 'FAIL'}")
print(f"  mean={noise_result['mean_noise_mean']:.2f}, std={noise_result['mean_noise_std']:.2f}")
print(f"Verify upload:   {'PASS' if upload_result['passed'] else 'FAIL'}")
print(f"  {upload_result['actual_count']}/{upload_result['expected_count']} files")
print("=" * 60)
print("All checks passed!" if (noise_result['passed'] and upload_result['passed']) else "SOME CHECKS FAILED")